In [1]:
# This script is used to process the data from a polarization filter calibration experiment. It assumes the data is stored in a .sif file with 2D images for each angle of the polarization filter.

import pickle
from pathlib import Path

import sif_parser
import numpy as np
import matplotlib.pyplot as plt

base_loc = r"D:\experiment 23 mrt 2026"
# f_pickle = Path(rf"{base_loc}\labctl\S1_044_calibr_80to140_0.4000s_idx.pkl")
f_pickle = Path(rf"{base_loc}\S1_calibr_0to360_0.5000s_idx.pkl")
f_data = Path(rf"{base_loc}\laser_height2.sif")  # Replace with path of the .sif file
has_background = False

# Load pickle file
info = pickle.load(open(f_pickle, "rb"))
data, img_info = sif_parser.np_open(f_data)
image_size = img_info['size']

def get_data(data, info, index):
    # Get the keys for the signal and background indices
    # Get the indices for the signal and background
    sig_ind = info['indices'][0][index][0] + index*5
    bg_ind = info['indices'][0][index][1]

    if len(bg_ind) != 0:
        # Get the data for the signal and background
        sig_data = data[sig_ind, :, :]
        bg_data = data[bg_ind, :, :]

        sig_data = np.squeeze(sig_data)
        bg_data_avg = np.squeeze(np.mean(bg_data, axis=0))
        return sig_data - bg_data_avg
    else:
        sig_data = data[sig_ind, :, :]
        sig_data = np.squeeze(sig_data)
        return sig_data

# Get the used angles from the pickle info
if "polarization_angle" in info:
    degs = info["polarization_angle"]
elif "x" in info:
    degs = info["x"]
else:
    raise ValueError("No angle information found in pickle file.")

# Extract the data from the .sif file
n_frames = info["N_frames"][0]
for n in info["N_frames"][1:]:
    if n != n_frames:
        raise ValueError("Number of frames varies over measurements")

indexes = [375, 385]

sig = np.zeros((degs.size, n_frames, image_size[1], indexes[1]-indexes[0]))

for i in range(len(degs)):
    sigi = get_data(data, info, i)
    sig[i] = sigi[:, :, slice(*indexes)]
sigs = np.mean(sig, axis=2)
sigs_avg = np.max(sigs, axis=2)

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\experiment 23 mrt 2026\\S1_calibr_0to360_0.5000s_idx.pkl'

In [ ]:
fig, ax = plt.subplots()
ax.imshow(sig[0, 0])

fig, ax = plt.subplots()
ax.plot(sigs_avg)

In [ ]:
# Fit (degs, vib_sig) with  y = A + B*cos(x*pi/180-C)^2  (which is Malus' law)
from scipy.optimize import curve_fit

def fit_func(x, A, B, C):
    return A + B*np.cos(x*np.pi/180-C)**2

# Select a range of interest for the fit
# Usually, the signal doesn't look like a perfect sine. This can be because the filter has a slight tilt, such that the focal point might shift a bit.
deg_rep = np.repeat(degs, n_frames)
vib_flat = sigs_avg.flatten()
print(deg_rep.shape)
print(vib_flat.shape, sigs_avg.shape, sigs.shape)
idx_fit = (deg_rep > 30) & (deg_rep < 200)
popt, pcov = curve_fit(fit_func, deg_rep[idx_fit], vib_flat[idx_fit], p0=[100, 10_000, 0.6*np.pi], bounds=([0, 0, 0], [np.inf, np.inf, np.pi]))

x = np.linspace(0, 360, 100)
y = fit_func(x, *popt)

B_max = popt[1] + popt[0]
C_max = popt[2] % np.pi

ang_max = C_max*180/np.pi
print(ang_max)

plt.plot(x, y, 'r-')
plt.plot(deg_rep, vib_flat, 'ko', alpha=0.5)
plt.plot(ang_max, B_max, 'ro', label=rf'Max: {ang_max:.2f} $\pm {np.sqrt(pcov[2, 2]):.4f} \degree $')
plt.xlabel("Rotation [deg]")
plt.ylabel("Intensity")
plt.legend(loc='upper right')
plt.show()

print(popt, pcov)

In [ ]:
data_med2 = np.mean(sig, axis=(3,))
normalized_intensity = data_med2 / np.sum(data_med2, axis=2)[:, :, None]
rows = np.arange(image_size[1])[None, None, :]
average = np.sum(rows*normalized_intensity, axis=2)
deg_rep = np.repeat(degs, n_frames)

plt.figure()
plt.plot(deg_rep, average.flatten(), 'ko', alpha=0.5)
plt.ylabel("Laser location [px]")
plt.xlabel("Rotation [deg]")
plt.grid()